# LeetCode 801 — Minimum Swaps To Make Sequences Increasing bằng Python

- Link: https://leetcode.com/problems/minimum-swaps-to-make-sequences-increasing/
- Độ khó: **Hard**
- Chủ đề: Quy hoạch động (DP hai trạng thái), Memoization, Tối ưu bộ nhớ O(1), Đồ thị ràng buộc

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def minSwap(self, nums1: List[int], nums2: List[int]) -> int:
```

## 1. Giới thiệu & mục tiêu học

Có hai mảng cùng độ dài. Ở mỗi vị trí `i`, ta được phép **đổi chỗ** `nums1[i]` với `nums2[i]`. Cần ít nhất bao nhiêu lần đổi để **cả hai** mảng đều **tăng ngặt**? Đề đảm bảo luôn làm được.

Đây là bài mẫu cho kiểu **DP có trạng thái phụ**: quyết định ở vị trí `i` chỉ phụ thuộc vào việc vị trí `i−1` **có đổi hay không**. Ta có hai dãy DP song song `keep[i]` và `swap[i]`, và cuối cùng chỉ cần 4 biến.

Sau notebook này bạn sẽ:

1. Viết brute force thử mọi tập vị trí đổi (`2ⁿ`) để đối chiếu.
2. Nhận ra **trạng thái** đúng: `(i, vị trí i có đổi không)`.
3. Viết DP **top-down** (đệ quy + memo) và **bottom-up** (bảng).
4. Tối ưu xuống **O(1) bộ nhớ** với hai biến lăn.
5. Hiểu hai điều kiện `c1`/`c2` và vì sao chúng có thể **cùng đúng**.

## 2. Đề bài & ví dụ

**Phát biểu lại.** Cho hai mảng số nguyên `nums1`, `nums2` cùng độ dài `n`. Một **thao tác**: chọn chỉ số `i`, đổi chỗ `nums1[i]` và `nums2[i]`. Trả về số thao tác ít nhất để cả `nums1` và `nums2` **tăng ngặt** (`arr[0] < arr[1] < ... < arr[n-1]`). Dữ liệu đảm bảo luôn có cách.

**Ví dụ 1**

```text
Input: nums1 = [1,3,5,4], nums2 = [1,2,3,7]
Output: 1
Giải thích: đổi ở i = 3 → nums1 = [1,3,5,7], nums2 = [1,2,3,4].
```

**Ví dụ 2**

```text
Input: nums1 = [0,3,5,8,9], nums2 = [2,1,4,6,9]
Output: 1
Giải thích: đổi ở i = 1 → nums1 = [0,1,5,8,9], nums2 = [2,3,4,6,9].
```

**Ràng buộc**

- `2 <= nums1.length <= 10⁵`
- `nums2.length == nums1.length`
- `0 <= nums1[i], nums2[i] <= 2 · 10⁵`
- Dữ liệu đảm bảo có đáp án.

In [1]:
from typing import List

EXAMPLES = [
    ([1, 3, 5, 4], [1, 2, 3, 7], 1),
    ([0, 3, 5, 8, 9], [2, 1, 4, 6, 9], 1),
    ([1, 2], [3, 4], 0),
    ([3, 2], [1, 4], 1),
]

def strictly_increasing(a):
    return all(a[i] < a[i + 1] for i in range(len(a) - 1))

for a, b, exp in EXAMPLES:
    print(a, b, "-> kỳ vọng", exp)

[1, 3, 5, 4] [1, 2, 3, 7] -> kỳ vọng 1
[0, 3, 5, 8, 9] [2, 1, 4, 6, 9] -> kỳ vọng 1
[1, 2] [3, 4] -> kỳ vọng 0
[3, 2] [1, 4] -> kỳ vọng 1


## 3. Kiến thức nền

### 3.1. Kiểm tra mảng tăng ngặt

`all(a[i] < a[i+1] for i in range(n-1))` — chú ý **ngặt** (`<`), không phải `<=`.

In [2]:
print(strictly_increasing([1, 3, 5, 7]), strictly_increasing([1, 3, 3, 7]))

True False


### 3.2. Duyệt mọi tập con bằng bitmask

Với `n` vị trí, mỗi số `mask` từ `0` tới `2ⁿ − 1` biểu diễn một tập con: bit `i` bật ⇔ vị trí `i` được chọn. `bin(mask).count('1')` là kích thước tập con.

In [3]:
n = 3
for mask in range(1 << n):
    chosen = [i for i in range(n) if mask >> i & 1]
    print(f"{mask:03b} -> {chosen}")

000 -> []
001 -> [0]
010 -> [1]
011 -> [0, 1]
100 -> [2]
101 -> [0, 2]
110 -> [1, 2]
111 -> [0, 1, 2]


### 3.3. Memoization với `functools.lru_cache`

Đệ quy có các trạng thái lặp lại → lưu kết quả lại. `@lru_cache(None)` tự lưu theo tham số. Lưu ý giới hạn đệ quy (~1000) khi `n` lớn → với `n = 10⁵` phải tăng giới hạn hoặc chuyển sang bottom-up.

In [4]:
from functools import lru_cache

@lru_cache(maxsize=None)
def fib(k):
    return k if k < 2 else fib(k - 1) + fib(k - 2)

print(fib(80))

23416728348467685


### 3.4. Vô cực trong Python

`float('inf')` lớn hơn mọi số; `min(inf, 3) = 3`. Dùng để khởi tạo trạng thái "chưa đạt được".

In [5]:
INF = float('inf')
print(min(INF, 3), INF + 1 == INF)

3 True


## 4. Ý tưởng nền tảng / trực giác

### 4.1. Chỉ hai vị trí kề nhau mới ràng buộc nhau

Hai mảng tăng ngặt ⇔ với mọi `i ≥ 1`: `A[i-1] < A[i]` và `B[i-1] < B[i]` (sau khi đổi). Mỗi điều kiện chỉ liên quan tới cặp `(i−1, i)`. → quyết định ở `i` chỉ cần biết quyết định ở `i−1`.

### 4.2. Hai điều kiện cho cặp `(i−1, i)`

Gọi `a, b = nums1, nums2` ban đầu.

- `c1`: `a[i-1] < a[i]` **và** `b[i-1] < b[i]` → hợp lệ nếu **cả hai** vị trí cùng giữ nguyên, hoặc **cả hai** cùng đổi (đổi cả hai thì các so sánh y hệt).
- `c2`: `a[i-1] < b[i]` **và** `b[i-1] < a[i]` → hợp lệ nếu **đúng một** trong hai vị trí đổi.

Có thể `c1` và `c2` **cùng đúng** (khi đó ta tự do), hoặc chỉ một cái đúng (bị ràng buộc). Đề đảm bảo ít nhất một cái đúng.

### 4.3. Tính tay ví dụ 1: `a = [1,3,5,4]`, `b = [1,2,3,7]`

| i | (a[i-1],a[i]) | (b[i-1],b[i]) | c1 | c2 |
|---|---|---|---|---|
| 1 | (1,3) | (1,2) | ✔ | `1<2` và `1<3` ✔ |
| 2 | (3,5) | (2,3) | ✔ | `3<3` ✘ |
| 3 | (5,4) | (3,7) | ✘ (5<4 sai) | `5<7` và `3<4` ✔ |

- `i = 2` chỉ `c1` → vị trí 1 và 2 phải "cùng trạng thái".
- `i = 3` chỉ `c2` → vị trí 2 và 3 phải "khác trạng thái".
- Phương án ít đổi nhất: giữ 0, 1, 2; đổi 3 → **1** lần.

In [6]:
def conditions(a, b):
    rows = []
    for i in range(1, len(a)):
        c1 = a[i - 1] < a[i] and b[i - 1] < b[i]
        c2 = a[i - 1] < b[i] and b[i - 1] < a[i]
        rows.append((i, c1, c2))
    return rows

for i, c1, c2 in conditions([1, 3, 5, 4], [1, 2, 3, 7]):
    print(f"i={i}: c1={c1}, c2={c2}")

i=1: c1=True, c2=True
i=2: c1=True, c2=False
i=3: c1=False, c2=True


## 5. Cách 1 — Brute Force: thử mọi tập vị trí đổi

**Ý tưởng.** Với mỗi `mask` trong `[0, 2ⁿ)`, đổi các vị trí có bit bật, kiểm tra hai mảng tăng ngặt, cập nhật đáp án nhỏ nhất theo số bit.

In [7]:
def min_swap_brute(a, b):
    n = len(a)
    best = float('inf')
    for mask in range(1 << n):
        A, B = a[:], b[:]
        for i in range(n):
            if mask >> i & 1:
                A[i], B[i] = B[i], A[i]
        if strictly_increasing(A) and strictly_increasing(B):
            best = min(best, bin(mask).count('1'))
    return best

for a, b, exp in EXAMPLES:
    print(min_swap_brute(a, b), "(kỳ vọng", exp, ")")
    assert min_swap_brute(a, b) == exp

1 (kỳ vọng 1 )
1 (kỳ vọng 1 )
0 (kỳ vọng 0 )
1 (kỳ vọng 1 )


**Độ phức tạp.** `O(2ⁿ · n)` thời gian. Chỉ dùng được với `n ≤ 15` hoặc tương đương.

**Nút thắt.** Brute force thử mọi tổ hợp dù ràng buộc chỉ là **cục bộ** (giữa hai vị trí kề nhau). Nhiều tổ hợp chia sẻ cùng "tiền tố" → lặp lại công việc → DP.

## 6. Bước đột phá

**Trạng thái DP.**
- `keep[i]` = số lần đổi ít nhất cho tiền tố `0..i` (cả hai tiền tố tăng ngặt) **khi vị trí `i` không đổi**.
- `swap[i]` = tương tự **khi vị trí `i` có đổi**.

**Chuyển trạng thái** (với `i ≥ 1`):

- *Nếu* `c1` *thì*: `keep[i] ← keep[i-1]` (cả hai giữ), `swap[i] ← swap[i-1] + 1` (cả hai đổi).
- *Nếu* `c2` *thì*: `keep[i] ← swap[i-1]` (trước đổi, nay giữ), `swap[i] ← keep[i-1] + 1` (trước giữ, nay đổi).
- Lấy `min` khi cả hai điều kiện cùng đúng.

**Khởi tạo:** `keep[0] = 0`, `swap[0] = 1`. **Đáp án:** `min(keep[n-1], swap[n-1])`.

**Invariant:** sau khi xử lý `i`, `keep[i]` và `swap[i]` đúng bằng tối ưu cho tiền tố `0..i` với trạng thái tương ứng ở `i`.

## 7. Các cách tiếp theo

### 7.1. Cách 2 — DP top-down (đệ quy + memo)

`go(i, prev_swapped)` = số lần đổi ít nhất cho các vị trí `i..n-1`, biết vị trí `i−1` có đổi hay không. Tại `i`, thử "giữ" và "đổi", chỉ nhận lựa chọn tương thích với `i−1`.

In [8]:
import sys

def min_swap_memo(a, b):
    n = len(a)
    sys.setrecursionlimit(max(10000, 4 * n + 100))

    @lru_cache(maxsize=None)
    def go(i, prev_swapped):
        if i == n:
            return 0
        # giá trị ở i-1 sau quyết định trước đó
        pa, pb = (b[i - 1], a[i - 1]) if prev_swapped else (a[i - 1], b[i - 1])
        best = float('inf')
        if pa < a[i] and pb < b[i]:          # giữ nguyên i
            best = min(best, go(i + 1, False))
        if pa < b[i] and pb < a[i]:          # đổi i
            best = min(best, 1 + go(i + 1, True))
        return best

    return min(go(1, False), 1 + go(1, True))

for a, b, exp in EXAMPLES:
    assert min_swap_memo(a, b) == exp
print("Top-down memo: OK")

Top-down memo: OK


Code nộp LeetCode cho cách 2 (chú ý `n` tới `10⁵` → đệ quy sâu; trên LeetCode có thể cần tăng giới hạn đệ quy, và vẫn có nguy cơ tràn ngăn xếp C — vì vậy cách bottom-up được ưa chuộng hơn):

In [9]:
class SolutionMemo:
    def minSwap(self, nums1: List[int], nums2: List[int]) -> int:
        n = len(nums1)
        sys.setrecursionlimit(max(10000, 4 * n + 100))
        memo = {}

        def go(i, sw):
            if i == n:
                return 0
            key = (i, sw)
            if key in memo:
                return memo[key]
            pa, pb = (nums2[i - 1], nums1[i - 1]) if sw else (nums1[i - 1], nums2[i - 1])
            res = float('inf')
            if pa < nums1[i] and pb < nums2[i]:
                res = go(i + 1, 0)
            if pa < nums2[i] and pb < nums1[i]:
                res = min(res, 1 + go(i + 1, 1))
            memo[key] = res
            return res

        return min(go(1, 0), 1 + go(1, 1))

for a, b, exp in EXAMPLES:
    assert SolutionMemo().minSwap(a, b) == exp
print("SolutionMemo: OK")

SolutionMemo: OK


**Độ phức tạp cách 2:** `2n` trạng thái × `O(1)` → **`O(n)`** thời gian, `O(n)` bộ nhớ (memo + ngăn xếp đệ quy).

### 7.2. Cách 3 — DP bottom-up với hai mảng

In [10]:
def min_swap_table(a, b):
    n = len(a)
    INF = float('inf')
    keep = [INF] * n
    swap = [INF] * n
    keep[0], swap[0] = 0, 1
    for i in range(1, n):
        if a[i - 1] < a[i] and b[i - 1] < b[i]:          # c1
            keep[i] = min(keep[i], keep[i - 1])
            swap[i] = min(swap[i], swap[i - 1] + 1)
        if a[i - 1] < b[i] and b[i - 1] < a[i]:          # c2
            keep[i] = min(keep[i], swap[i - 1])
            swap[i] = min(swap[i], keep[i - 1] + 1)
    return min(keep[-1], swap[-1])

for a, b, exp in EXAMPLES:
    assert min_swap_table(a, b) == exp
print("Bottom-up bảng: OK")

Bottom-up bảng: OK


In [11]:
class SolutionTable:
    def minSwap(self, nums1: List[int], nums2: List[int]) -> int:
        n = len(nums1)
        INF = float('inf')
        keep = [INF] * n
        swap = [INF] * n
        keep[0], swap[0] = 0, 1
        for i in range(1, n):
            if nums1[i - 1] < nums1[i] and nums2[i - 1] < nums2[i]:
                keep[i] = keep[i - 1]
                swap[i] = swap[i - 1] + 1
            if nums1[i - 1] < nums2[i] and nums2[i - 1] < nums1[i]:
                keep[i] = min(keep[i], swap[i - 1])
                swap[i] = min(swap[i], keep[i - 1] + 1)
        return min(keep[-1], swap[-1])

for a, b, exp in EXAMPLES:
    assert SolutionTable().minSwap(a, b) == exp
print("SolutionTable: OK")

SolutionTable: OK


**Độ phức tạp cách 3:** `O(n)` thời gian, `O(n)` bộ nhớ. Ưu điểm: có bảng đầy đủ để **truy vết** phương án.

### 7.3. Cách 4 — DP bộ nhớ O(1)

`keep[i]`, `swap[i]` chỉ dùng `keep[i-1]`, `swap[i-1]` → giữ 2 biến, tính 2 biến mới mỗi vòng.

In [12]:
def min_swap_o1(a, b):
    keep, swap = 0, 1
    for i in range(1, len(a)):
        nk = ns = float('inf')
        if a[i - 1] < a[i] and b[i - 1] < b[i]:
            nk, ns = keep, swap + 1
        if a[i - 1] < b[i] and b[i - 1] < a[i]:
            nk, ns = min(nk, swap), min(ns, keep + 1)
        keep, swap = nk, ns
    return min(keep, swap)

for a, b, exp in EXAMPLES:
    assert min_swap_o1(a, b) == exp
print("DP O(1): OK")

DP O(1): OK


### 7.4. Cách 5 — Góc nhìn đồ thị: các "khối liên kết"

Một cách nhìn khác rất đẹp: giữa hai vị trí kề nhau:
- chỉ `c1` đúng → hai vị trí **phải cùng trạng thái** (cùng đổi hoặc cùng giữ);
- chỉ `c2` đúng → **phải khác trạng thái**;
- cả hai đúng → **không ràng buộc** (cắt khối tại đây).

Vậy dãy chia thành các **khối** được nối bởi các ràng buộc "cùng/khác". Trong mỗi khối, chọn trạng thái vị trí đầu thì mọi vị trí khác bị xác định. Có 2 lựa chọn: số vị trí đổi là `x` hoặc `len − x`. Lấy `min` cho từng khối và cộng lại — các khối độc lập vì giữa chúng không có ràng buộc.

In [13]:
def min_swap_blocks(a, b):
    n = len(a)
    total = 0
    parity = [0]            # trạng thái tương đối so với vị trí đầu khối
    for i in range(1, n + 1):
        if i < n:
            c1 = a[i - 1] < a[i] and b[i - 1] < b[i]
            c2 = a[i - 1] < b[i] and b[i - 1] < a[i]
        if i == n or (c1 and c2):
            x = sum(parity)                       # số vị trí "khác đầu khối"
            total += min(x, len(parity) - x)
            parity = [0]
        elif c1:
            parity.append(parity[-1])             # cùng trạng thái
        else:
            parity.append(parity[-1] ^ 1)         # khác trạng thái
    return total

for a, b, exp in EXAMPLES:
    assert min_swap_blocks(a, b) == exp
print("Khối liên kết: OK")

Khối liên kết: OK


Ở cách 5, `x` = số vị trí có trạng thái khác vị trí đầu khối. Nếu đầu khối **giữ**, số lần đổi trong khối là `x`; nếu đầu khối **đổi**, số lần đổi là `len − x`. Độ phức tạp `O(n)`.

## 8. TRACE từng bước

Trace cách 3 trên ví dụ 2: `a = [0,3,5,8,9]`, `b = [2,1,4,6,9]`.

In [14]:
def trace_table(a, b):
    n = len(a)
    INF = float('inf')
    keep, swap = [INF] * n, [INF] * n
    keep[0], swap[0] = 0, 1
    print(f"i=0: keep=0 swap=1")
    for i in range(1, n):
        c1 = a[i - 1] < a[i] and b[i - 1] < b[i]
        c2 = a[i - 1] < b[i] and b[i - 1] < a[i]
        if c1:
            keep[i] = min(keep[i], keep[i - 1]); swap[i] = min(swap[i], swap[i - 1] + 1)
        if c2:
            keep[i] = min(keep[i], swap[i - 1]); swap[i] = min(swap[i], keep[i - 1] + 1)
        print(f"i={i}: a[i-1..i]=({a[i-1]},{a[i]}) b[i-1..i]=({b[i-1]},{b[i]}) "
              f"c1={c1!s:5} c2={c2!s:5} -> keep={keep[i]} swap={swap[i]}")
    # truy vết phương án: đi ngược, tìm trạng thái trước p tương thích và cho đúng giá trị
    dp = [keep, swap]
    plan = [0] * n
    state = 0 if keep[-1] <= swap[-1] else 1
    for i in range(n - 1, 0, -1):
        plan[i] = state
        c1 = a[i - 1] < a[i] and b[i - 1] < b[i]
        c2 = a[i - 1] < b[i] and b[i - 1] < a[i]
        for p in (0, 1):
            ok = c1 if p == state else c2
            if ok and dp[p][i - 1] + state == dp[state][i]:
                state = p
                break
    plan[0] = state
    print("đáp án =", min(keep[-1], swap[-1]), "| phương án (1 = đổi):", plan)
    A = [b[i] if plan[i] else a[i] for i in range(n)]
    B = [a[i] if plan[i] else b[i] for i in range(n)]
    print("sau khi đổi:", A, B)
    return min(keep[-1], swap[-1])

trace_table([0, 3, 5, 8, 9], [2, 1, 4, 6, 9])

i=0: keep=0 swap=1
i=1: a[i-1..i]=(0,3) b[i-1..i]=(2,1) c1=False c2=True  -> keep=1 swap=1
i=2: a[i-1..i]=(3,5) b[i-1..i]=(1,4) c1=True  c2=True  -> keep=1 swap=2
i=3: a[i-1..i]=(5,8) b[i-1..i]=(4,6) c1=True  c2=True  -> keep=1 swap=2
i=4: a[i-1..i]=(8,9) b[i-1..i]=(6,9) c1=True  c2=True  -> keep=1 swap=2
đáp án = 1 | phương án (1 = đổi): [1, 0, 0, 0, 0]
sau khi đổi: [2, 3, 5, 8, 9] [0, 1, 4, 6, 9]


1

Bảng trạng thái:

| i | c1 | c2 | keep[i] | swap[i] | giải thích |
|---|---|---|---|---|---|
| 0 | — | — | 0 | 1 | khởi tạo |
| 1 | ✘ (2<1 sai) | ✔ (0<1, 2<3) | swap[0] = 1 | keep[0]+1 = 1 | phải khác trạng thái |
| 2 | ✔ | ✔ (3<4, 1<5) | min(1, 1) = 1 | min(2, 2) = 2 | tự do |
| 3 | ✔ | ✔ | min(1, 2) = 1 | min(3, 2) = 2 | tự do |
| 4 | ✔ | ✔ (8<9, 6<9) | min(1, 2) = 1 | min(3, 2) = 2 | tự do |

Đáp án `min(keep[4], swap[4]) = 1`: chỉ cần đổi vị trí 1 (hoặc tương đương đổi vị trí 0 — cả hai đều 1 lần; truy vết chọn một phương án).

### Trace cách 5 (khối liên kết) trên ví dụ 1

In [15]:
def trace_blocks(a, b):
    n = len(a)
    for i in range(1, n):
        c1 = a[i - 1] < a[i] and b[i - 1] < b[i]
        c2 = a[i - 1] < b[i] and b[i - 1] < a[i]
        rel = "tự do (cắt khối)" if c1 and c2 else ("cùng" if c1 else "khác")
        print(f"giữa {i-1} và {i}: {rel}")
    print("đáp án:", min_swap_blocks(a, b))

trace_blocks([1, 3, 5, 4], [1, 2, 3, 7])

giữa 0 và 1: tự do (cắt khối)
giữa 1 và 2: cùng
giữa 2 và 3: khác
đáp án: 1


Ví dụ 1: khối đầu là `{0}` (cắt giữa 0 và 1 vì tự do), khối thứ hai `{1, 2, 3}` với quan hệ "cùng, khác" → parity `[0, 0, 1]` → `x = 1`, `min(1, 2) = 1`. Tổng `0 + 1 = 1`.

## 9. Code nộp LeetCode

In [16]:
class Solution:
    def minSwap(self, nums1: List[int], nums2: List[int]) -> int:
        keep, swap = 0, 1                      # vị trí 0: giữ tốn 0, đổi tốn 1
        for i in range(1, len(nums1)):
            nk = ns = float('inf')
            a0, a1, b0, b1 = nums1[i - 1], nums1[i], nums2[i - 1], nums2[i]
            if a0 < a1 and b0 < b1:            # cùng trạng thái với i-1
                nk, ns = keep, swap + 1
            if a0 < b1 and b0 < a1:            # khác trạng thái với i-1
                nk, ns = min(nk, swap), min(ns, keep + 1)
            keep, swap = nk, ns
        return min(keep, swap)

for a, b, exp in EXAMPLES:
    assert Solution().minSwap(a, b) == exp
print("Solution: OK")

Solution: OK


## 10. Vì sao thuật toán đúng

**Cấu trúc con tối ưu.** Xét phương án tối ưu cho tiền tố `0..i` với trạng thái `s_i` ở vị trí `i`. Phần `0..i−1` của nó là một phương án hợp lệ cho tiền tố `0..i−1` với trạng thái `s_{i−1}`, và chi phí = chi phí phần trước + `[s_i = đổi]`. Nếu phần trước không tối ưu, thay bằng phương án tối ưu với cùng `s_{i−1}` vẫn hợp lệ (vì tính hợp lệ của cặp `(i−1, i)` chỉ phụ thuộc `s_{i−1}, s_i`) và rẻ hơn — mâu thuẫn.

**Tính hợp lệ của cặp.** Sau quyết định, cặp giá trị ở `i−1` là `(a[i−1], b[i−1])` hoặc hoán đổi; ở `i` tương tự. Có 4 tổ hợp:
- (giữ, giữ) và (đổi, đổi): cần `a[i−1] < a[i]`, `b[i−1] < b[i]` → `c1`.
- (giữ, đổi) và (đổi, giữ): cần `a[i−1] < b[i]`, `b[i−1] < a[i]` → `c2`.

Công thức chuyển trạng thái liệt kê đúng 4 tổ hợp này. Do đó `keep[i]`, `swap[i]` tính đúng theo quy nạp, và đáp án là `min` của hai trạng thái cuối.

**Cách 5.** Ràng buộc giữa các vị trí tạo thành một đường; cạnh "tự do" không ràng buộc, nên các khối độc lập. Trong khối, ràng buộc "cùng/khác" xác định duy nhất trạng thái mọi vị trí theo vị trí đầu (2 lựa chọn). Đề đảm bảo có nghiệm → mọi cạnh có ít nhất một loại. Tổng các `min` là tối ưu.

## 11. Phân tích Time/Space

| Cách | Time | Space |
|---|---|---|
| 1. Brute force | `O(2ⁿ · n)` | `O(n)` |
| 2. Top-down memo | `O(n)` | `O(n)` (+ ngăn xếp) |
| 3. Bottom-up bảng | `O(n)` | `O(n)` |
| 4. DP O(1) | `O(n)` | `O(1)` |
| 5. Khối liên kết | `O(n)` | `O(n)` (có thể O(1)) |

In [17]:
import random, time

def random_case(n, rnd, vmax=None):
    """Sinh hai dãy tăng ngặt rồi đổi ngẫu nhiên -> luôn có nghiệm."""
    vmax = vmax or 4 * n + 10
    A = sorted(rnd.sample(range(vmax), n))
    B = sorted(rnd.sample(range(vmax), n))
    for i in range(n):
        if rnd.random() < 0.5:
            A[i], B[i] = B[i], A[i]
    return A, B

rnd = random.Random(801)
a_big, b_big = random_case(10**5, rnd, 2 * 10**5)
# (không chạy bản đệ quy ở n = 1e5: dễ vượt giới hạn đệ quy / ngăn xếp C)
for name, fn in [("table", min_swap_table), ("o1", min_swap_o1), ("blocks", min_swap_blocks),
                 ("Solution", lambda x, y: Solution().minSwap(x, y))]:
    t0 = time.perf_counter()
    r = fn(a_big, b_big)
    print(f"{name:8}: {r}  ({(time.perf_counter() - t0) * 1000:.1f} ms)")

table   : 49206  (38.7 ms)


o1      : 49206  (27.5 ms)


blocks  : 49206  (24.4 ms)
Solution: 49206  (26.7 ms)


## 12. Bộ test

In [18]:
methods = {
    "brute": min_swap_brute,
    "memo": min_swap_memo,
    "table": min_swap_table,
    "o1": min_swap_o1,
    "blocks": min_swap_blocks,
    "SolutionMemo": lambda a, b: SolutionMemo().minSwap(a, b),
    "SolutionTable": lambda a, b: SolutionTable().minSwap(a, b),
    "Solution": lambda a, b: Solution().minSwap(a, b),
}

# 12.1 Ví dụ + edge cases
edge = EXAMPLES + [
    ([1, 2], [1, 2], 0),
    ([5, 2, 3], [1, 6, 7], 1),               # đổi vị trí 0
    ([0, 4, 4, 5, 9], [0, 1, 6, 8, 10], 1),
    ([3, 3, 8, 9, 10], [1, 7, 4, 6, 8], 1),
    ([0, 7, 8, 10, 10, 11, 12, 13, 19, 18], [4, 4, 5, 7, 11, 14, 15, 16, 17, 20], 4),
    ([2, 1, 6, 9], [0, 4, 5, 7], 1),
]
for a, b, exp in edge:
    ref = min_swap_brute(a, b)
    if exp is not None:
        assert ref == exp, (a, b, ref)
    for name, fn in methods.items():
        assert fn(a, b) == ref, (name, a, b)
print("Ví dụ + edge cases: OK")

# 12.2 Ngẫu nhiên có seed, n nhỏ: đối chiếu brute force
for seed in range(800):
    r = random.Random(seed)
    n = r.randint(2, 11)
    a, b = random_case(n, r, vmax=r.choice([2 * n + 2, 3 * n, 6 * n]))
    ref = min_swap_brute(a, b)
    for name, fn in methods.items():
        assert fn(a, b) == ref, (name, a, b)
print("800 bộ ngẫu nhiên: khớp brute force")

# 12.3 n lớn: các cách DP khớp nhau
for seed in range(10):
    r = random.Random(9000 + seed)
    a, b = random_case(r.randint(500, 2000), r)
    vals = {name: fn(a, b) for name, fn in methods.items() if name != "brute"}
    assert len(set(vals.values())) == 1, vals
print("n lớn: các cách khớp nhau")

Ví dụ + edge cases: OK


800 bộ ngẫu nhiên: khớp brute force
n lớn: các cách khớp nhau


## 13. Lỗi người mới hay gặp

### Lỗi 1 — Dùng `if ... elif` thay vì hai `if` độc lập

**Code sai:**

In [19]:
def wrong_elif(a, b):
    keep, swap = 0, 1
    for i in range(1, len(a)):
        nk = ns = float('inf')
        if a[i - 1] < a[i] and b[i - 1] < b[i]:
            nk, ns = keep, swap + 1
        elif a[i - 1] < b[i] and b[i - 1] < a[i]:     # SAI: bỏ qua khi c1 cũng đúng
            nk, ns = swap, keep + 1
        keep, swap = nk, ns
    return min(keep, swap)

a, b = [0, 3, 5, 8, 9], [2, 1, 4, 6, 9]
print("sai:", wrong_elif(a, b), " đúng:", Solution().minSwap(a, b))
a, b = [4, 5, 4, 9], [0, 1, 8, 9]
print("sai:", wrong_elif(a, b), " đúng:", Solution().minSwap(a, b))

sai: 1  đúng: 1
sai: 2  đúng: 1


**Vì sao sai:** ví dụ đầu tình cờ cho kết quả đúng, nhưng với `[4,5,4,9], [0,1,8,9]` code sai trả 2 thay vì 1. `c1` và `c2` có thể **cùng đúng** (ở đây tại cặp `(2,3)`); khi đó cả 4 tổ hợp đều hợp lệ và ta phải lấy `min` của chúng. `elif` bỏ mất các tổ hợp "khác trạng thái".

**Code đúng:** hai `if` độc lập, nhánh thứ hai dùng `min`.

### Lỗi 2 — Cập nhật `keep` rồi dùng `keep` mới để tính `swap`

**Code sai:**

In [20]:
def wrong_inplace(a, b):
    keep, swap = 0, 1
    for i in range(1, len(a)):
        c1 = a[i - 1] < a[i] and b[i - 1] < b[i]
        c2 = a[i - 1] < b[i] and b[i - 1] < a[i]
        if c1 and c2:
            keep = min(keep, swap)
            swap = min(swap, keep) + 1          # SAI: keep đã bị ghi đè
        elif c1:
            swap = swap + 1
        else:
            keep, swap = swap, keep + 1
    return min(keep, swap)

for a, b, _ in EXAMPLES + [([0, 4, 4, 5, 9], [0, 1, 6, 8, 10], 1)]:
    print("sai:", wrong_inplace(a, b), " đúng:", Solution().minSwap(a, b))

sai: 1  đúng: 1
sai: 1  đúng: 1
sai: 0  đúng: 0
sai: 1  đúng: 1
sai: 1  đúng: 1


**Vì sao sai:** ở trường hợp tự do, `swap[i] = min(keep[i-1], swap[i-1]) + 1` — cần **giá trị cũ**. Ghi đè `keep` trước thì `swap` vẫn tình cờ đúng ở đây (vì `min(min(k,s), s) = min(k,s)`), nhưng thói quen cập nhật tại chỗ rất dễ sai khi công thức phức tạp hơn. Ví dụ đảo thứ tự: tính `swap` trước rồi `keep = min(keep, swap)` thì `keep` dùng `swap` **mới** (đã `+1`) — có thể sai:

In [21]:
def wrong_inplace2(a, b):
    keep, swap = 0, 1
    for i in range(1, len(a)):
        c1 = a[i - 1] < a[i] and b[i - 1] < b[i]
        c2 = a[i - 1] < b[i] and b[i - 1] < a[i]
        if c1 and c2:
            swap = min(swap, keep) + 1
            keep = min(keep, swap)              # SAI: dùng swap mới
        elif c1:
            swap = swap + 1
        else:
            keep, swap = swap, keep + 1
    return min(keep, swap)

cnt_bad = 0
for seed in range(300):
    r = random.Random(seed)
    a, b = random_case(r.randint(2, 9), r)
    if wrong_inplace2(a, b) != Solution().minSwap(a, b):
        cnt_bad += 1
print("số bộ test sai với wrong_inplace2:", cnt_bad, "/ 300")

số bộ test sai với wrong_inplace2: 22 / 300


**Code đúng:** tính `nk, ns` từ `keep, swap` cũ rồi gán đồng thời `keep, swap = nk, ns`.

### Lỗi 3 — Khởi tạo `swap[0] = 0`

**Code sai:**

In [22]:
def wrong_init(a, b):
    keep, swap = 0, 0                           # SAI: đổi vị trí 0 cũng tốn 1
    for i in range(1, len(a)):
        nk = ns = float('inf')
        if a[i - 1] < a[i] and b[i - 1] < b[i]:
            nk, ns = keep, swap + 1
        if a[i - 1] < b[i] and b[i - 1] < a[i]:
            nk, ns = min(nk, swap), min(ns, keep + 1)
        keep, swap = nk, ns
    return min(keep, swap)

a, b = [3, 2], [1, 4]
print("sai:", wrong_init(a, b), " đúng:", Solution().minSwap(a, b))

sai: 0  đúng: 1


**Vì sao sai:** trạng thái "vị trí 0 đổi" đã tốn **1** thao tác. Với `[3,2],[1,4]`, đổi vị trí 0 là cách tối ưu (1 lần) nhưng code sai đếm 0.

**Code đúng:** `keep, swap = 0, 1`.

## 14. Mẹo & nhận dạng pattern

- **"Ở mỗi vị trí có 2 lựa chọn, ràng buộc chỉ giữa hai vị trí kề nhau"** → DP với trạng thái `(i, lựa chọn ở i)`; số trạng thái nhỏ → O(1) bộ nhớ.
- Liệt kê **tất cả tổ hợp** (giữ/đổi × giữ/đổi) và điều kiện của từng tổ hợp; dùng các `if` **độc lập**.
- **Cập nhật đồng thời** khi DP lăn (`a, b = f(a, b), g(a, b)`).
- Góc nhìn **ràng buộc cùng/khác** = tô 2 màu trên đường → mỗi khối 2 phương án.
- Cùng họ: **1187 Make Array Strictly Increasing** (khó hơn: thay bằng phần tử của mảng khác), **926 Flip String to Monotone Increasing**, **198 House Robber** (DP 2 trạng thái).

## 15. Biến thể

1. **Không tăng ngặt mà không giảm** (`≤`): chỉ đổi `<` thành `<=` trong `c1`, `c2`.
2. **Ba mảng, mỗi vị trí được hoán vị 3 giá trị tuỳ ý:** trạng thái = hoán vị nào (6 trạng thái).
3. **Chi phí đổi khác nhau ở mỗi vị trí** `cost[i]`: thay `+1` bằng `+cost[i]`.
4. **Có thể không có nghiệm:** trả `-1` nếu cả hai trạng thái là `inf`.
5. **In ra phương án:** lưu bảng và truy vết (xem mục 8 và bài tập 4).

Minh hoạ biến thể 3 + 4:

In [23]:
def min_cost_swap(a, b, cost):
    INF = float('inf')
    keep, swap = 0, cost[0]
    for i in range(1, len(a)):
        nk = ns = INF
        if a[i - 1] < a[i] and b[i - 1] < b[i]:
            nk, ns = keep, swap + cost[i]
        if a[i - 1] < b[i] and b[i - 1] < a[i]:
            nk, ns = min(nk, swap), min(ns, keep + cost[i])
        keep, swap = nk, ns
    res = min(keep, swap)
    return -1 if res == INF else res

print(min_cost_swap([1, 3, 5, 4], [1, 2, 3, 7], [5, 5, 5, 5]))   # đổi i=3: 5
print(min_cost_swap([1, 3, 5, 4], [1, 2, 3, 7], [1, 1, 1, 9]))   # đổi 0,1,2 rẻ hơn? -> kiểm tra
print(min_cost_swap([1, 1], [1, 1], [1, 1]))                     # không có nghiệm -> -1

5
2
-1


## 16. Bảng so sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Brute force | thử mọi tập đổi | `O(2ⁿ n)` | `O(n)` | đối chiếu |
| 2. Top-down memo | `go(i, prev_swapped)` | `O(n)` | `O(n)` | dễ nghĩ ra từ đệ quy |
| 3. Bottom-up bảng | `keep[i]`, `swap[i]` | `O(n)` | `O(n)` | cần truy vết |
| 4. DP O(1) | 2 biến lăn | `O(n)` | `O(1)` | **nộp bài** |
| 5. Khối liên kết | ràng buộc cùng/khác, min mỗi khối | `O(n)` | `O(1)`–`O(n)` | hiểu sâu cấu trúc |

## 17. Bài tập

**Bài 1 (đoán output).** `minSwap([5, 2, 3], [1, 6, 7])` và `minSwap([1, 2, 6, 7], [3, 4, 5, 8])` = ?

**Bài 2 (tính tay).** Lập bảng `keep/swap` cho `nums1 = [2, 1, 6, 9]`, `nums2 = [0, 4, 5, 7]`.

**Bài 3 (điền TODO).** Hoàn thiện `min_swap_nondecreasing(a, b)` cho biến thể "không giảm" (`≤`).

**Bài 4 (tự viết lại).** Viết `min_swap_plan(a, b)` trả về **danh sách chỉ số cần đổi** (một phương án tối ưu), và kiểm tra áp dụng nó cho ra hai mảng tăng ngặt.

**Bài 5 (mini challenge).** Ba mảng `a, b, c`; ở mỗi vị trí được hoán vị tuỳ ý bộ ba `(a[i], b[i], c[i])` với chi phí = 0 nếu giữ nguyên, 1 nếu khác. Tìm chi phí nhỏ nhất để cả ba mảng tăng ngặt (hoặc `-1`).

In [24]:
# Bài 3 — TODO (không gọi khi chạy notebook)
def min_swap_nondecreasing(a, b):
    # TODO: như Solution nhưng dùng <= trong c1, c2
    pass

# Bài 4 — TODO
def min_swap_plan(a, b):
    # TODO: lưu bảng keep/swap + con trỏ trạng thái trước; truy vết từ cuối
    pass

# Bài 5 — TODO
def min_cost_three(a, b, c):
    # TODO: trạng thái = hoán vị (6 cái) của bộ ba ở vị trí i
    pass

## 18. Gợi ý 3 mức

**Bài 1.**
- Mức 1: tính `c1, c2` cho từng cặp.
- Mức 2: `[5,2,3],[1,6,7]`: giữa 0 và 1 chỉ `c2` đúng (`5<6`, `1<2`) → vị trí 0 và 1 khác trạng thái; giữa 1 và 2 chỉ `c1` → cùng trạng thái. Khối `{0,1,2}` parity `[0,1,1]` → `min(2, 1) = 1` (đổi vị trí 0).
- Mức 3: `[1,2,6,7],[3,4,5,8]`: cả hai mảng đã tăng ngặt → `0`.

**Bài 2.**
- Mức 1: `keep[0]=0, swap[0]=1`.
- Mức 2: `i=1`: `c1`: `2<1` sai; `c2`: `2<4`, `0<1` ✔ → `keep=1, swap=1`.
- Mức 3: tiếp tục với `i=2, 3`.

**Bài 3.**
- Mức 1: chỉ thay `<` bằng `<=`.
- Mức 2: ít nhất một điều kiện vẫn phải đúng (giả sử có nghiệm).
- Mức 3: kiểm tra bằng brute force với điều kiện `<=`.

**Bài 4.**
- Mức 1: lưu `from_keep[i]`, `from_swap[i]` = trạng thái trước tốt nhất.
- Mức 2: bắt đầu từ trạng thái cuối có giá trị nhỏ hơn.
- Mức 3: đi ngược, thu các `i` có trạng thái "đổi".

**Bài 5.**
- Mức 1: trạng thái `p ∈` 6 hoán vị của `(0,1,2)`.
- Mức 2: chuyển `p → q` hợp lệ nếu ba mảng đều tăng tại cặp `(i−1, i)`.
- Mức 3: `dp[i][q] = min_p dp[i−1][p] + [q ≠ identity]`.

## 19. Đáp án tham khảo

In [25]:
S = Solution()
# Bài 1
assert S.minSwap([5, 2, 3], [1, 6, 7]) == 1
assert S.minSwap([1, 2, 6, 7], [3, 4, 5, 8]) == 0

# Bài 2
assert min_swap_table([2, 1, 6, 9], [0, 4, 5, 7]) == min_swap_brute([2, 1, 6, 9], [0, 4, 5, 7]) == 1

# Bài 3
def min_swap_nondecreasing_ans(a, b):
    keep, swap = 0, 1
    for i in range(1, len(a)):
        nk = ns = float('inf')
        if a[i - 1] <= a[i] and b[i - 1] <= b[i]:
            nk, ns = keep, swap + 1
        if a[i - 1] <= b[i] and b[i - 1] <= a[i]:
            nk, ns = min(nk, swap), min(ns, keep + 1)
        keep, swap = nk, ns
    return min(keep, swap)

def brute_nondecreasing(a, b):
    n = len(a); best = float('inf')
    for mask in range(1 << n):
        A, B = a[:], b[:]
        for i in range(n):
            if mask >> i & 1:
                A[i], B[i] = B[i], A[i]
        if all(A[i] <= A[i + 1] for i in range(n - 1)) and all(B[i] <= B[i + 1] for i in range(n - 1)):
            best = min(best, bin(mask).count('1'))
    return best

for seed in range(300):
    r = random.Random(seed)
    n = r.randint(2, 9)
    A = sorted(r.randint(0, 6) for _ in range(n)); B = sorted(r.randint(0, 6) for _ in range(n))
    for i in range(n):
        if r.random() < 0.5:
            A[i], B[i] = B[i], A[i]
    assert min_swap_nondecreasing_ans(A, B) == brute_nondecreasing(A, B)

# Bài 4
def min_swap_plan_ans(a, b):
    n = len(a)
    INF = float('inf')
    dp = [[INF, INF] for _ in range(n)]      # dp[i][s]
    par = [[None, None] for _ in range(n)]
    dp[0] = [0, 1]
    for i in range(1, n):
        c1 = a[i - 1] < a[i] and b[i - 1] < b[i]
        c2 = a[i - 1] < b[i] and b[i - 1] < a[i]
        for s in (0, 1):
            for p in (0, 1):
                ok = c1 if p == s else c2
                if ok and dp[i - 1][p] + s < dp[i][s]:
                    dp[i][s] = dp[i - 1][p] + s
                    par[i][s] = p
    s = 0 if dp[-1][0] <= dp[-1][1] else 1
    plan = []
    for i in range(n - 1, -1, -1):
        if s:
            plan.append(i)
        if i:
            s = par[i][s]
    return sorted(plan)

for seed in range(300):
    r = random.Random(seed)
    a, b = random_case(r.randint(2, 12), r)
    plan = min_swap_plan_ans(a, b)
    A, B = a[:], b[:]
    for i in plan:
        A[i], B[i] = B[i], A[i]
    assert strictly_increasing(A) and strictly_increasing(B)
    assert len(plan) == S.minSwap(a, b)

# Bài 5
from itertools import permutations
def min_cost_three_ans(a, b, c):
    n = len(a)
    perms = list(permutations(range(3)))
    INF = float('inf')
    def vals(i, p):
        t = (a[i], b[i], c[i])
        return [t[p[0]], t[p[1]], t[p[2]]]
    dp = {p: (0 if p == (0, 1, 2) else 1) for p in perms}
    for i in range(1, n):
        nd = {}
        for q in perms:
            vq = vals(i, q)
            best = INF
            for p in perms:
                vp = vals(i - 1, p)
                if all(vp[k] < vq[k] for k in range(3)):
                    best = min(best, dp[p])
            nd[q] = best + (0 if q == (0, 1, 2) else 1)
        dp = nd
    res = min(dp.values())
    return -1 if res == INF else res

def brute_three(a, b, c):
    from itertools import product
    n = len(a)
    perms = list(permutations(range(3)))
    best = INF
    for choice in product(perms, repeat=n):
        rows = [[(a[i], b[i], c[i])[choice[i][k]] for i in range(n)] for k in range(3)]
        if all(strictly_increasing(r) for r in rows):
            best = min(best, sum(ch != (0, 1, 2) for ch in choice))
    return -1 if best == INF else best

for seed in range(150):
    r = random.Random(seed)
    n = r.randint(1, 4)
    a = [r.randint(0, 9) for _ in range(n)]; b = [r.randint(0, 9) for _ in range(n)]; c = [r.randint(0, 9) for _ in range(n)]
    assert min_cost_three_ans(a, b, c) == brute_three(a, b, c), (a, b, c)
print("Tất cả đáp án đúng")

Tất cả đáp án đúng


## 20. Template nhớ nhanh + Checklist trước khi submit

**Template:**

```python
keep, swap = 0, 1
for i in range(1, n):
    nk = ns = inf
    if a[i-1] < a[i] and b[i-1] < b[i]:   # cùng trạng thái
        nk, ns = keep, swap + 1
    if a[i-1] < b[i] and b[i-1] < a[i]:   # khác trạng thái
        nk, ns = min(nk, swap), min(ns, keep + 1)
    keep, swap = nk, ns
return min(keep, swap)
```

**Checklist trước khi submit:**

- [ ] `swap` khởi tạo bằng **1**.
- [ ] Hai `if` **độc lập** (không `elif`).
- [ ] Tính giá trị mới từ giá trị **cũ**, gán đồng thời.
- [ ] Dùng `<` (tăng **ngặt**).
- [ ] Không đệ quy sâu với `n = 10⁵` (ưu tiên bottom-up).